# Importiamo le librerie

In [ ]:
SEED = 42
import pandas as pd
import numpy as np
import gensim
from tqdm import tqdm
import matplotlib.pyplot as plt
import nltk
import pickle
import joblib

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
from nltk.stem import WordNetLemmatizer

import gensim
from gensim.models import LdaModel
from gensim import corpora
from gensim import models

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings

# Inizio seconda parte

In [ ]:
def carica_dati(file_csv):     # Carica dati se vuoi scrivere la descrizione del libro
    data = pd.read_csv(file_csv)
    data.drop(columns=['Unnamed: 0'],inplace = True)
    data['text_split'] = data['text'].str.split()
    dictionary = corpora.Dictionary(data['text_split'])
    corpus = [dictionary.doc2bow(text) for text in data['text_split']]
    return data, dictionary, corpus


In [ ]:
def carica_dati2(file_csv,titolo):  # Carica dati se vuoi scrivere il titolo
    data = pd.read_csv(file_csv)
    data.drop(columns=['Unnamed: 0'],inplace = True)
    lista_titoli = list(data.Book_title)

    if titolo in lista_titoli:
        print('Libro trovato, ne cerco altri simili...')
        data['text_split'] = data['text'].str.split()           # In questa parte si cerca di estrarre il testo della varibile Plot
        ll = data[data.Book_title == titolo].text_split.tolist()
        l2 = data[data.Book_title == titolo].text.tolist()
        frase_sporca = str(l2[0])                   # Bisogna capire come ottenere la frase (in questa parte dava un warning ma nulla di preoccupante)
        frase = ll[0]
        d = data[data.Book_title != titolo]
        d['text_split'] = d['text'].str.split()
        dictionary = corpora.Dictionary(d['text_split'])
        corpus = [dictionary.doc2bow(text) for text in d['text_split']]

        num = 1     # Ecco le possbili alternative se si inserisce il libro sbagliato
    elif titolo not in lista_titoli:
        print('Libro non trovato')
        num = 2
    else:
        print('Qualcosa è andato storto, ci dispiace')
        num = 3

    return d, dictionary, corpus, num, frase, frase_sporca

# Importiamo il modello LDA dalla prima parte

In [ ]:
def carica_modello(p_modello):
    model = LdaModel.load(p_modello)
    return model

Implementiamo ora una funzone che ci permetta di lavorare sulla nostra frase di ricerca:

In [ ]:
def get_lemma(token):      # Sono le funzioni che servono poi per fare il prepocessing
    return WordNetLemmatizer().lemmatize(token)
def is_stopwords(token):
    if token in stop_words:
        return True
    return False
def tokenize(text):
    tokens = nltk.word_tokenize(text)
    return tokens

In [ ]:
def soel(frase,oo):    # Fare il preprocess
    frase_di_soel = tokenize(frase)
    frase_di_soel = [token.lower() for token in frase_di_soel]
    frase_di_soel = [token for token in frase_di_soel if len(token) > 3]
    frase_di_soel = [token for token in frase_di_soel if token not in stop_words]
    oconel = [get_lemma(token) for token in frase_di_soel]
    if oo == 1:
        oconel = dictionary.doc2bow(oconel)
    elif oo == 2:
        pass
    return oconel

In [ ]:
def soellino(frase,oo):       # Nel codice viene inserito un numero (al posto di oo) solo nel caso di cosa_vuoi_fare == 2
    if oo == 1:               # e serve solo a trasformare la frase nel formato doc2bow
        oconel = dictionary.doc2bow(frase)
    elif oo == 2:
        pass
    return oconel

In [ ]:
def ottieni_topic(frase):
    lista_topic_frase = model.get_document_topics(frase)  # Restituisce la lista con tutti i topic e le rispettive probabilità di appartenenza al topic
    maggiore = ['niente',0.00]                            # Con il ciclo for andiamo ad ottenre la coppia id topic con probabilità più alta
    for topic in lista_topic_frase:
        id_topic = int(topic[0])
        prob = float(topic[1])
        if prob > maggiore[1]:
            maggiore = [id_topic, prob]
    print('Id_topic più probabile è: ',maggiore[0],'\nCon probabilità: ',maggiore[1])
    return maggiore

In [ ]:
def legge_zipf(frase, num_indi):   #num_indi = numero di libri che si vogliono ricevere in output, frase= descrizione inserita dall'utente
    vectorizer = TfidfVectorizer(min_df = 20)  # inizializziamo avectorizer
    tfidf_matrix = vectorizer.fit_transform(f.text) # otteniamo la matrice dei punteggi tf-idf

    frase = ' '.join(frase) #Stringhiamo la lista
    stringa_input_tfidf = vectorizer.transform([frase])  # trasformiamo la frase in un formato simile alla mtrice
    similarity_scores = cosine_similarity(stringa_input_tfidf, tfidf_matrix).flatten()   # Calcoliamo la cosine similarity
    similarity_indices = similarity_scores.argsort()[::-1]  # ordiniamo i punteggi coseno e otteniamo gli indici
    f['cosine_similarity'] = similarity_scores   # creiamo la  colonna relativa ai punteggi coseno
    libri_simili = f.iloc[similarity_indices]    # teniamo i primi n libri simili


    indi = libri_simili.head(num_indi).index

    return indi,stringa_input_tfidf,similarity_scores, f

In [ ]:
def data_recom(indici,):         # Solo per abbellire l'output e creare 2 dataset. Uno contiene solo alcune colonne (quelle importanti)
    df = dd.loc[indici]          # l'altro contiene tutte le varibili, compresa la cosine
    dff = df.drop(columns=['topic_words','rilevance','best_topic','text','text_split','cosine_similarity'])
    return df, dff

In [ ]:
############################################################################################################################

In [ ]:
############################################################################################################################

In [ ]:
############################################################################################################################

In [ ]:
##frase = ' a hobbit who sets out on a perilous quest to destroy the One Ring and prevent the Dark Lord Sauron from conquering Middle-earth. With a fellowship of companions, including Gandalf, Aragorn, and Legolas, they face daunting challenges and confront the forces of evil. Through their trials and sacrifices, they strive to save their world from darkness and restore peace.'
#numero_simili = 3
#data, dictionary, corpus = carica_dati('data_finale.csv')
#model = carica_modello('modello_finale.gensim')
#frase_pulita = soel(frase,1)
#caba = ottieni_topic(frase_pulita)
#f = data[data.best_topic == caba[0]]
#indici_racomandati,stringa_input_tfidf,similarity_scores, dd = legge_zipf(soel(frase,2), numero_simili)
#Data_host, raccomandati = data_recom(indici_racomandati)

# Inizio
Possiamo usare tutto ciò che abbiamo programmato

In [ ]:
print('Premere 1 se vuoi trovare un libro utilizzando una breve descrizione')
print('Premere 2 se vuoi trovare un libro partendo dal titolo')
cosa_fare = int(input('Cosa vuoi fare? '))
warnings.filterwarnings("ignore")           # esce un warning che mi consiglia di usare un modo diverso tagliare il dataset (nel punto 2)

if cosa_fare == 1:
    frase = str(input('Scrivi la descrizione: '))
    numero_simili = int(input('Inserisci il numero di libri simili che vuoi cercare: '))
    data, dictionary, corpus = carica_dati('data_finale22.csv')   # Il codice carica i dati che verranno usati solo nell' if cosa fare == 1
    model = carica_modello('modello_finale22.gensim')
    frase_pulita = soel(frase,1)        # La funzione soel, fa il preprocessing alla frase in input
    caba = ottieni_topic(frase_pulita)  # La funzione ottieni_topic restituisce il topic più probabile di apparteneza della frase in input
    f = data[data.best_topic == caba[0]]  # teniamo solo le righe dove la colonna best_topic == al migliore (chiamato 'caba[0]')
    indici_racomandati,stringa_input_tfidf,similarity_scores, dd = legge_zipf(soel(frase,2), numero_simili)  # parte di word embedding
    Data_host, raccomandati = data_recom(indici_racomandati)

elif cosa_fare == 2:
    titolo = str(input('Scrivi il titolo del libro: '))
    numero_simili = int(input('Inserisci il numero di libri simili che vuoi cercare: '))
    data, dictionary, corpus, opz, frase_pulita, frase_non_pulita = carica_dati2('data_finale22.csv',titolo)
    if opz == 1:
        model = carica_modello('modello_finale22.gensim')
        fra = soellino(frase_pulita,1)          # serve solo a prendere il titolo del libro e trasformarolo in un formato utilizzabile (impacchetato)
        caba = ottieni_topic(fra)               # La funzione ottieni_topic restituisce il topic più probabile di apparteneza della frase in input
        print(caba)
        f = data[data.best_topic == caba[0]]    # teniamo solo le righe dove la colonna best_topic == al migliore (chiamato 'caba[0]')
        indici_racomandati,stringa_input_tfidf,similarity_scores, dd = legge_zipf(soel(frase_non_pulita,2), numero_simili)
        Data_host, raccomandati = data_recom(indici_racomandati)

    elif opz == 2:                  # Queste sono solo le possibili alternative che restituiscono i motivi di errore
        print("E' possibile che il titolo del libro inserito non sia presente nella nostra libreria o che sia stato scritto in modo incorretto")
        print('Riprovare controllando, grazie.')

    elif opz == 3:
        print('Stiamo lavorando al suo problema, torni più tardi...')

    else:
        pass
raccomandati

Premere 1 se vuoi trovare un libro utilizzando una breve descrizione
Premere 2 se vuoi trovare un libro partendo dal titolo
Cosa vuoi fare? 2
Scrivi il titolo del libro: Blade Runner 2: The Edge of Human
Inserisci il numero di libri simili che vuoi cercare: 4
Libro trovato, ne cerco altri simili...
Id_topic più probabile è:  12 
Con probabilità:  0.37168994545936584
[12, 0.37168994545936584]


,Book_title,Author,Publication_date,Book_genres,Plot_summary
15693,Barrayar,Lois McMaster Bujold,1991,"['Science Fiction', 'Speculative fiction', 'Fa...","As Barrayar begins, Aral and Cordelia Vorkosi..."
8,Blade Runner 3: Replicant Night,K. W. Jeter,1996-10-01,"['Science Fiction', 'Speculative fiction']","Living on Mars, Deckard is acting as a consul..."
921,When Harlie Was One,David Gerrold,1972,"['Science Fiction', 'Speculative fiction', 'Fi...",Central to the story is an Artificial Intelli...
4641,Gridiron,Philip Kerr,1995,"['Science Fiction', 'Speculative fiction', 'Fi...",Ray Richardson and his top team of architects...


In [ ]:
Data_host

,text,best_topic,rilevance,topic_words,Book_title,Author,Publication_date,Book_genres,Plot_summary,text_split,cosine_similarity
15693,barrayar begin aral cordelia vorkosigan expect...,12,0.390656,"[('human', 0.002709936), ('world', 0.002539190...",Barrayar,Lois McMaster Bujold,1991,"['Science Fiction', 'Speculative fiction', 'Fa...","As Barrayar begins, Aral and Cordelia Vorkosi...","[barrayar, begin, aral, cordelia, vorkosigan, ...",0.333580
8,live mar deckard act consult movi crew film st...,12,0.351682,"[('human', 0.002709936), ('world', 0.002539190...",Blade Runner 3: Replicant Night,K. W. Jeter,1996-10-01,"['Science Fiction', 'Speculative fiction']","Living on Mars, Deckard is acting as a consul...","[live, mar, deckard, act, consult, movi, crew,...",0.304602
921,central_stori artifici_intellig name also_refe...,12,0.757820,"[('human', 0.002709936), ('world', 0.002539190...",When Harlie Was One,David Gerrold,1972,"['Science Fiction', 'Speculative fiction', 'Fi...",Central to the story is an Artificial Intelli...,"[central_stori, artifici_intellig, name, also_...",0.219076
4641,ray richardson top team architect develop supe...,12,0.451379,"[('human', 0.002709936), ('world', 0.002539190...",Gridiron,Philip Kerr,1995,"['Science Fiction', 'Speculative fiction', 'Fi...",Ray Richardson and his top team of architects...,"[ray, richardson, top, team, architect, develo...",0.153133


In [ ]:
# Titoli d'esempio:
#   Blade Runner 2: The Edge of Human
#   The Lord of the Rings
#   Animal Farm
#   The Plague
#   Children of Dune